# Notebook Colab pour entrainer le modele ECG

Notebook pour ouvrir dans VS Code avec l'extension Colab ou dans un runtime Google Colab. Il prepare l'environnement, verifie le GPU, lance l'entrainement du pipeline fusion, puis affiche les artefacts sauvegardes.

## 1. Installation et configuration de l'extension Google Colab dans VS Code

Ouvre ce notebook avec l'extension Colab dans VS Code. La cellule suivante installe les dependances du projet et place le notebook dans la racine du depot.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / 'requirements.txt').exists() and (candidate / 'config.yaml').exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError('Impossible de trouver la racine du projet.')

os.chdir(project_root)
print(f'Project root: {project_root}')
print(f'Python: {sys.executable}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')], check=True)
print('Dependances installees ou deja presentes.')

Project root: C:\Users\User\Downloads\ecg_data (1)\ecg_data\projet
Python: c:\Users\User\Downloads\ecg_data (1)\ecg_data\projet\.venv\Scripts\python.exe
Dependances installees ou deja presentes.


## 2. Connexion a l'environnement Colab

Cette section verifie le runtime, l'acces au GPU et la presence du dataset. Si tu es sur Colab distant, assure-toi que le depot et les donnees sont accessibles.

In [2]:
import logging
import json

import numpy as np
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU non disponible, execution CPU.')

csv_path_check = project_root / 'PTB-XL ECG dataset' / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1' / 'ptbxl_database.csv'
print(f'config.yaml exists: {(project_root / "config.yaml").exists()}')
print(f'dataset csv exists: {csv_path_check.exists()}')

CUDA available: False
GPU non disponible, execution CPU.
config.yaml exists: True
dataset csv exists: True


## 3. Preparation des donnees d'entrainement

Cette section charge les metadonnees PTB-XL, cree les splits et prepare les jeux d'entrainement, de validation et de test.

In [3]:
from common.utils import set_seed, load_config, CLASS_NAMES, NUM_CLASSES
from pipeline_fusion.dataset import load_fusion_dataset, ECGDualDataset
from torch.utils.data import DataLoader

set_seed(42)
config = load_config(str(project_root / 'config.yaml'))

csv_path = project_root / config['paths']['ptbxl_signals'] / 'ptbxl_database.csv'
signals_dir = project_root / config['paths']['ptbxl_signals']
images_dir = project_root / config['paths']['ptbxl_images']

logger = logging.getLogger('colab_notebook')
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

full_df = load_fusion_dataset(str(csv_path), str(signals_dir), str(images_dir), logger)
train_df = full_df[full_df['strat_fold'].isin([1, 2, 3, 4, 5, 6, 7, 8])].reset_index(drop=True)
val_df = full_df[full_df['strat_fold'] == 9].reset_index(drop=True)
test_df = full_df[full_df['strat_fold'] == 10].reset_index(drop=True)

train_ds = ECGDualDataset(train_df, augment=True)
val_ds = ECGDualDataset(val_df, augment=False)
test_ds = ECGDualDataset(test_df, augment=False)

batch_size = 4
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

class_counts = np.stack(train_df['target'].values).sum(axis=0)
print('Classes:', CLASS_NAMES)
print('Train size:', len(train_ds), 'Val size:', len(val_ds), 'Test size:', len(test_ds))
print('Class counts:', class_counts.tolist())

INFO:colab_notebook:Chargement CSV: C:\Users\User\Downloads\ecg_data (1)\ecg_data\projet\PTB-XL ECG dataset\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1\ptbxl_database.csv
INFO:colab_notebook:  21837 enregistrements
Chargement: 100%|██████████| 21837/21837 [00:18<00:00, 1193.46it/s]
INFO:colab_notebook:  Signaux manquants: 0
INFO:colab_notebook:  Sans label valide: 1319
INFO:colab_notebook:  Enregistrements valides: 20518


Classes: ['NORM', 'MI', 'STTC', 'CD', 'ARR']
Train size: 16412 Val size: 2042 Test size: 2064
Class counts: [7553.0, 3315.0, 3944.0, 3905.0, 1191.0]


## 4. Definition du modele

Cette section instancie le modele fusion avec le backbone signal, le backbone image, la co-attention et les metadonnees cliniques.

In [4]:
from pipeline_fusion.model import ECGFusionModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ECGFusionModel(
    num_classes=NUM_CLASSES,
    embed_dim=256,
    pretrained=True,
    dropout=0.3,
    modality_drop_p=0.15,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(model.__class__.__name__)

Model parameters: 34,922,841
ECGFusionModel


## 5. Entrainement du modele

Cette cellule est maintenant prete pour un run Colab GPU complet.

Reglages:
- `RUN_PROFILE = "full_colab"` pour un vrai entrainement long
- `RUN_PROFILE = "smoke"` pour un test rapide

In [7]:
from pipeline_fusion.trainer import FusionTrainer

trainer_config = {
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'patience': 10,
}

trainer = FusionTrainer(model, trainer_config, device, logger)
trainer.setup_loss(class_counts, loss_type='logit_adj', tau=1.0)

# full_colab: entrainement long pour runtime GPU Colab.
# smoke: test rapide local.
RUN_PROFILE = 'full_colab'  # options: 'full_colab', 'smoke'

if RUN_PROFILE == 'full_colab':
    epochs = 30
    batch_size = 16 if torch.cuda.is_available() else 4
    max_batches = None
    num_workers = 2 if torch.cuda.is_available() else 0
else:
    epochs = 1
    batch_size = 2
    max_batches = 1
    num_workers = 0

if RUN_PROFILE == 'full_colab' and not torch.cuda.is_available():
    print('Attention: profil full_colab active sans GPU. Passe le runtime Colab en GPU pour un vrai entrainement.')

ckpt_dir = project_root / 'models' / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

history = trainer.train_loop(
    train_loader,
    val_loader,
    epochs=epochs,
    ckpt_dir=str(ckpt_dir),
    start_epoch=1,
    max_batches=max_batches,
)

print('Run profile:', RUN_PROFILE)
print('epochs:', epochs, 'batch_size:', batch_size, 'max_batches:', max_batches)
print('Training history keys:', list(history.keys()))

INFO:colab_notebook:Loss: MultiLabelLogitAdjustedBCE (τ=1.0)
INFO:colab_notebook:  Poids par classe: {'NORM': '1.279', 'MI': '2.237', 'STTC': '2.012', 'CD': '2.024', 'ARR': '3.964'}
INFO:colab_notebook:  α (aux 1D+2D): 0.3
INFO:colab_notebook:  β_cd: 0.2
INFO:colab_notebook:  β_mi: 0.3
INFO:colab_notebook:Entraînement: epochs 1→30, patience=10
INFO:colab_notebook:Multi-task: L_fus + 0.3*L_1d + 0.3*L_2d + 0.2*L_cd + 0.3*L_mi
INFO:colab_notebook:
INFO:colab_notebook:Epoch 1/30


Attention: profil full_colab active sans GPU. Passe le runtime Colab en GPU pour un vrai entrainement.


Train:   1%|          | 30/4103 [07:02<18:38:46, 16.48s/it, loss=2.7235, lr=3.00e-04]

: 

## 6. Evaluation et sauvegarde du modele

Cette section recharge le meilleur checkpoint, evalue le modele sur le test set et rappelle ou se trouvent les artefacts enregistres.

In [6]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix

best_ckpt_path = ckpt_dir / 'best_fusion_model.pth'
if best_ckpt_path.exists():
    best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'], strict=False)
    trainer.model = model.to(device)

    test_metrics, preds, labels = trainer.validate(test_loader)
    print('Test metrics:')
    print(json.dumps(test_metrics, indent=2, default=str))
    print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))
    print('Confusion matrices:')
    for idx, matrix in enumerate(multilabel_confusion_matrix(labels, preds)):
        print(CLASS_NAMES[idx], matrix.tolist())
else:
    print(f'Checkpoint not found: {best_ckpt_path}')

print(f'Best checkpoint: {best_ckpt_path}')
print(f'Run manifest: {ckpt_dir / "run_manifest.json"}')
print(f'Training metrics: {ckpt_dir / "training_metrics.jsonl"}')

Test metrics:
{
  "loss": 1.652884899414787,
  "accuracy": 0.005329457364341085,
  "f1_macro": 0.14828470891074946,
  "pr_auc_macro": 0.2488,
  "f1_per_class": {
    "NORM": 0.1396,
    "MI": 0.0802,
    "STTC": 0.385,
    "CD": 0.0079,
    "ARR": 0.1287
  },
  "recall_per_class": {
    "NORM": 0.0901,
    "MI": 0.048,
    "STTC": 1.0,
    "CD": 0.004,
    "ARR": 1.0
  }
}
              precision    recall  f1-score   support

        NORM       0.31      0.09      0.14       955
          MI       0.24      0.05      0.08       417
        STTC       0.24      1.00      0.38       492
          CD       0.33      0.00      0.01       498
         ARR       0.07      1.00      0.13       142

   micro avg       0.17      0.30      0.21      2504
   macro avg       0.24      0.43      0.15      2504
weighted avg       0.28      0.30      0.15      2504
 samples avg       0.16      0.28      0.19      2504

Confusion matrices:
NORM [[918, 191], [869, 86]]
MI [[1585, 62], [397, 20]]
STTC 